# Qutrit Post-Processing: Choi Matrix Reconstruction and Fidelity Evaluation

This notebook post-processes the raw qutrit process tomography data collected by
`SimulationQutrit.ipynb` to produce the process fidelities reported in the manuscript.

For each of the 40 metric operators, and for both the raw and the REM-corrected counts,
the notebook:
1. Loads the qutrit count data from `data/raw_tomography/qutrit_raw_counts/`.
2. Clips any residual negative values from REM and renormalises each tomography setting.
3. Reconstructs the Choi matrix via linear inversion. The survival probability of each
   input state is the average of its estimates from the X-, Y- and Z-basis settings.
4. Projects the Choi matrix onto the set of completely positive, trace-non-increasing maps.
5. Computes the process fidelity against the ideal Choi matrix and compares against
   the adversarial threshold.

Results are saved to `data/processed/experimental_fidelities_qutrit.csv`. The last cell
combines them with the two-qubit results from `PostProcessingTwoQubit.ipynb` into
`data/processed/experimental_fidelities_master.csv`, so run that notebook first.

Run the notebook from the repository root.

In [ ]:
import sys
import csv
import os
from math import pi

import numpy as np
import pandas as pd
import scipy as sp
from scipy import linalg as la

from forest.benchmarking.tomography import (
    _pauli_process_tomo_settings,
    pauli2matrix,
    proj_choi_to_physical,
    state2matrix,
    unvec,
    vec,
)
from forest.benchmarking.observable_estimation import *
from forest.benchmarking.operator_tools import superoperator_transformations as supt

if sys.version_info < (3, 7):
    from pyquil.external.dataclasses import dataclass
else:
    from dataclasses import dataclass

## Metric operator parameter grid

In [ ]:
def eta_grid():
    """Generate a grid of 40 metric operator parameter vectors [r, theta, phi, alpha].

    The first 20 points sample two delta values at five nu values with phi = -pi/2.
    The next 20 points are related to the first 20 by a Hadamard transform, with
    phi = pi/2 and theta = pi/2.

    Returns:
        list of [r, theta, phi, alpha] parameter vectors (length 40).
    """
    grid = []
    nu_array = np.linspace(0.05, 0.95, 5)

    # First 20 points: phi = -pi/2 and pi/2, theta varies with delta
    for delta_array, phi in [
        (np.linspace(-pi/4, pi/4, 2), -pi/2),
        (np.linspace(3*pi/4, 5*pi/4, 2), pi/2),
    ]:
        for nu in nu_array:
            for delta in delta_array:
                r = (1 - nu) / (1 + nu)
                theta = pi/2 - delta if phi == -pi/2 else -pi/2 + delta
                alpha = 2 * np.arccos(np.sqrt(1 - np.abs(r)))
                grid.append([r, theta, phi, alpha])

    # Last 20 points: theta = pi/2, phi varies with delta (Hadamard-related metrics)
    delta_array = np.linspace(pi/4, 7*pi/4, 4)
    for nu in nu_array:
        for delta in delta_array:
            r = (1 - nu) / (1 + nu)
            theta = pi / 2
            phi = 3*pi/2 - delta
            alpha = 2 * np.arccos(np.sqrt(1 - np.abs(r)))
            grid.append([r, theta, phi, alpha])

    return grid

## Helper functions

In [ ]:
def get_etahalf(r, theta, phi):
    """Construct the square root of the metric operator eta from its parameters.

    Args:
        r: radial metric parameter.
        theta: polar angle parameter (radians).
        phi: azimuthal angle parameter (radians).

    Returns:
        eta_half: (2, 2) complex array, the matrix square root of the metric operator.
    """
    eta_half = np.array([
        [
            np.cos(theta/2)**2 + np.sqrt(r) * np.sin(theta/2)**2,
            (np.cos(phi) - 1j*np.sin(phi)) * np.cos(theta/2) * np.sin(theta/2) * (1 - np.sqrt(r))
        ],
        [
            (np.cos(phi) + 1j*np.sin(phi)) * np.cos(theta/2) * np.sin(theta/2) * (1 - np.sqrt(r)),
            np.sqrt(r) * np.cos(theta/2)**2 + np.sin(theta/2)**2
        ]
    ])
    return eta_half


def get_idealchoi(eta_half):
    """Compute the ideal Choi matrix for the IP-change channel defined by eta_half.

    Args:
        eta_half: (2, 2) complex array, the matrix square root of the metric operator.

    Returns:
        choi: (4, 4) complex array, the ideal Choi matrix.
    """
    max_entangled = np.array([[1, 0, 0, 1]]).T / np.sqrt(2)
    identity = np.eye(2)
    A = np.kron(identity, eta_half)
    choi = 2 * A @ max_entangled @ max_entangled.conj().T @ A.conj().T
    return choi


def get_fidelity(ideal_choi, estimated_choi):
    """Compute the process fidelity between the ideal and estimated Choi matrices.

    Uses the quantum fidelity formula F = Tr[sqrt(sqrt(A) B sqrt(A))] / (sqrt(Tr[A]) sqrt(Tr[B])).

    Args:
        ideal_choi: (4, 4) complex array, the ideal Choi matrix.
        estimated_choi: (4, 4) complex array, the experimentally reconstructed Choi matrix.

    Returns:
        fidelity: real scalar process fidelity.
    """
    sqrt_ideal = sp.linalg.sqrtm(ideal_choi)
    numerator = np.trace(sp.linalg.sqrtm(sqrt_ideal @ estimated_choi @ sqrt_ideal))
    denominator = np.sqrt(np.trace(ideal_choi)) * np.sqrt(np.trace(estimated_choi))
    return (numerator / denominator).real

## Build the linear inversion measurement matrix

The measurement matrix is constructed once from the standard Pauli process tomography
settings and is shared across all 40 metrics. It is augmented with identity rows to
fix the trace of the reconstructed Choi matrix.

In [ ]:
qs = [0]
tomo_expt_settings = list(_pauli_process_tomo_settings(qs))

# Input state density matrices for each preparation setting
# (ordered X+, X-, Y+, Y-, Z+, Z-; one per 3-observable block)
input_states = [
    state2matrix(tomo_expt_settings[i * 3].in_state, qs)
    for i in range(6)
]

# Measurement matrix from Pauli observables
measurement_matrix = np.vstack([
    vec(np.kron(
        state2matrix(setting.in_state, qs).conj(),
        pauli2matrix(setting.observable, qs)
    )).conj().T
    for setting in tomo_expt_settings
])

# Augment with identity rows to fix the trace (one per input state)
identity_rows = np.vstack([
    vec(np.kron(rho.conj(), np.eye(2))).conj().T
    for rho in input_states
])
measurement_matrix = np.vstack((measurement_matrix, identity_rows))

## Reconstruct Choi matrices and evaluate fidelities

For each metric, residual negative values from REM are clipped to zero and the counts
of each tomography setting renormalised before linear inversion.

Every input state is measured in the X, Y and Z bases, and each of these three settings
gives its own estimate of the survival probability $1 - p(|2\rangle)$. The estimates are
averaged over the three settings.

The reconstructed Choi matrix is then projected onto the set of completely positive,
trace-non-increasing maps before computing the process fidelity.

In [ ]:
INPUT_DIR = "data/raw_tomography/qutrit_raw_counts"
OUTPUT_DIR = "data/processed"
os.makedirs(OUTPUT_DIR, exist_ok=True)

grid = eta_grid()
measurement_pinv = np.linalg.pinv(measurement_matrix)


def fidelity_from_counts(counts, k):
    """Process fidelity for metric k from an (18, 3) array of qutrit outcome counts.

    Args:
        counts: counts in |0>, |1>, |2> for the 18 tomography settings, ordered as
            six input states (X+, X-, Y+, Y-, Z+, Z-) times three bases (X, Y, Z).
        k: metric index (0-39).

    Returns:
        fidelity: real scalar process fidelity with the ideal channel.
    """
    # Clip residual negative values and renormalise each setting
    probs = np.clip(np.asarray(counts, dtype=float), 0, None)
    probs = probs / probs.sum(axis=1, keepdims=True)

    # Pauli expectation value of each setting: p(|0>) - p(|1>)
    pauli_expectations = probs[:, 0] - probs[:, 1]

    # Survival probability 1 - p(|2>) of each input state, averaged over its
    # X-, Y- and Z-basis settings
    survival = (1 - probs[:, 2]).reshape(6, 3).mean(axis=1)

    expectations = np.concatenate([pauli_expectations, survival])[:, np.newaxis]

    # Reconstruct Choi matrix via linear inversion
    choi = unvec(measurement_pinv @ expectations)

    # Project onto completely positive, trace-non-increasing maps
    choi_proj = proj_choi_to_physical(choi, False)

    # Compute process fidelity against ideal Choi matrix
    eta_half = get_etahalf(grid[k][0], grid[k][1], grid[k][2])
    return get_fidelity(get_idealchoi(eta_half), choi_proj)


rows = []

for k in range(40):
    data_path = os.path.join(INPUT_DIR, f"qutrit_tomography_metric_{k+1:02d}.csv")
    data = pd.read_csv(data_path)

    fidelity_raw = fidelity_from_counts(data[['raw_0', 'raw_1', 'raw_2']].values, k)
    fidelity = fidelity_from_counts(
        data[['mitigated_0', 'mitigated_1', 'mitigated_2']].values, k)

    # Adversarial threshold for certifying a genuinely higher-dimensional process
    r = grid[k][0]
    threshold = (1 + np.sqrt(r)) / (np.sqrt(2) * np.sqrt(1 + r))

    rows.append({
        'index': k + 1,
        'r': r,
        'theta_pi': grid[k][1] / pi,
        'phi_pi': grid[k][2] / pi,
        'threshold_fidelity': round(threshold, 3),
        'qutrit_fidelity': round(fidelity, 3),
        'qutrit_passed': fidelity > threshold,
        'qutrit_raw_fidelity': round(fidelity_raw, 3),
        'qutrit_raw_passed': fidelity_raw > threshold,
    })

# Save results
qutrit_df = pd.DataFrame(rows)
out_path = os.path.join(OUTPUT_DIR, "experimental_fidelities_qutrit.csv")
qutrit_df.to_csv(out_path, index=False)
print(f"Done. Results saved to {out_path}")
print(f"Metrics passing threshold (REM-corrected): {qutrit_df['qutrit_passed'].sum()}/40")
print(f"Metrics passing threshold (raw): {qutrit_df['qutrit_raw_passed'].sum()}/40")

## Build the master table

Combines the qutrit results above with the two-qubit results written by
`PostProcessingTwoQubit.ipynb` into the summary table reported in the manuscript.

In [ ]:
qubit_path = os.path.join(OUTPUT_DIR, "experimental_fidelities_qubit.csv")

if not os.path.exists(qubit_path):
    print(f"{qubit_path} not found. Run PostProcessingTwoQubit.ipynb first.")
else:
    qubit_df = pd.read_csv(qubit_path)
    master_df = pd.DataFrame({
        'index': qutrit_df['index'],
        'r': qutrit_df['r'].round(3),
        'theta_pi': qutrit_df['theta_pi'].round(2),
        'phi_pi': qutrit_df['phi_pi'].round(2),
        'threshold_fidelity': qutrit_df['threshold_fidelity'],
        'qubit_fidelity': qubit_df['qubit_fidelity'],
        'qutrit_fidelity': qutrit_df['qutrit_fidelity'],
        'qubit_passed': qubit_df['qubit_passed'],
        'qutrit_passed': qutrit_df['qutrit_passed'],
        'qutrit_raw_fidelity': qutrit_df['qutrit_raw_fidelity'],
        'qutrit_raw_passed': qutrit_df['qutrit_raw_passed'],
    })
    master_path = os.path.join(OUTPUT_DIR, "experimental_fidelities_master.csv")
    master_df.to_csv(master_path, index=False)
    print(f"Master table saved to {master_path}")